<a href="https://colab.research.google.com/github/vikash-mahar/AI-MLNotes/blob/main/39Hyperparameter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from google.colab import files

In [3]:
uploaded = files.upload()

Saving diabetes.csv to diabetes.csv


In [5]:
df = pd.read_csv('diabetes.csv')

In [6]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [7]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [8]:
x=df.iloc[:,:-1].values
y=df.iloc[:,-1].values

In [9]:
x

array([[  6.   , 148.   ,  72.   , ...,  33.6  ,   0.627,  50.   ],
       [  1.   ,  85.   ,  66.   , ...,  26.6  ,   0.351,  31.   ],
       [  8.   , 183.   ,  64.   , ...,  23.3  ,   0.672,  32.   ],
       ...,
       [  5.   , 121.   ,  72.   , ...,  26.2  ,   0.245,  30.   ],
       [  1.   , 126.   ,  60.   , ...,  30.1  ,   0.349,  47.   ],
       [  1.   ,  93.   ,  70.   , ...,  30.4  ,   0.315,  23.   ]])

In [10]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [11]:
x= scaler.fit_transform(x)

In [12]:
x.shape

(768, 8)

In [13]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train, y_test=train_test_split(x,y,test_size=0.2,random_state=0)

In [14]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Dropout
!pip install -q -U keras-tuner
import keras_tuner as kt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.9 MB/s eta 0:00:00


In [19]:
def build_model(hp):
  model = Sequential()
  num_layers = hp.Int('num_layers', min_value=1, max_value=10)

  for i in range(num_layers):
    if i == 0:
      model.add(Dense(hp.Int('units_' + str(i), min_value=8, max_value=128, step=8),
                      activation=hp.Choice('activation_' + str(i), values=['relu','tanh','sigmoid']),
                      input_dim=8))
    else:
      model.add(Dense(hp.Int('units_' + str(i), min_value=8, max_value=128, step=8),
                      activation=hp.Choice('activation_' + str(i), values=['relu','tanh','sigmoid'])
                    ))
    model.add(Dropout(hp.Choice('dropout_' + str(i), values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))

  model.add(Dense(1,activation='sigmoid'))
  model.compile(optimizer= hp.Choice('optimizer',values=['rmsprop','sgd','adam','adadelta','nadam']),
                loss='binary_crossentropy',
                metrics=['accuracy'])
  return model

In [20]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials =3,
                        directory='mydir',
                        project_name='final'
                        )

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))

Trial 3 Complete [00h 02m 26s]
val_accuracy: 0.6948052048683167

Best val_accuracy So Far: 0.6948052048683167
Total elapsed time: 00h 02m 28s


In [23]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3,
 'units_0': 128,
 'activation_0': 'relu',
 'dropout_0': 0.4,
 'optimizer': 'sgd',
 'units_1': 32,
 'activation_1': 'sigmoid',
 'dropout_1': 0.1,
 'units_2': 112,
 'activation_2': 'sigmoid',
 'dropout_2': 0.5,
 'units_3': 120,
 'activation_3': 'relu',
 'dropout_3': 0.7,
 'units_4': 32,
 'activation_4': 'sigmoid',
 'dropout_4': 0.4}

In [24]:
model = tuner.get_best_models(num_models=1)[0]

In [26]:
model.fit(x_train,y_train,epochs=200,initial_epoch=6,validation_data=(x_test,y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.5635 - loss: 0.7357 - val_accuracy: 0.6948 - val_loss: 0.6172
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6010 - loss: 0.6936 - val_accuracy: 0.6948 - val_loss: 0.6198
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5847 - loss: 0.6957 - val_accuracy: 0.6948 - val_loss: 0.6158
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5733 - loss: 0.7211 - val_accuracy: 0.6948 - val_loss: 0.6191
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5733 - loss: 0.6936 - val_accuracy: 0.6948 - val_loss: 0.6160
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6010 - loss: 0.6970 - val_accuracy: 0.6948 - val_loss: 0.6156
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5879 - loss: 0.7309 - val_accuracy: 0.6948 - val_loss: 0.6228
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5993 - loss: 0.6883 - val_accuracy: